# Maia Lite 0.3.0 — Benchmark SFT v2.2 × v2.3

**Comparação controlada após reiniciar o Colab.** Recarrega o checkpoint-base original de 300.000 passos e os dois melhores adaptadores, sem treinamento. Avalia os mesmos prompts em dois formatos (`original` e `newline`), com geração greedy e amostragem controlada. Registra tokens, EOS, repetição e resultados para avaliação humana cega.

**Interpretação:** cada adaptador é testado em ambos os formatos, inclusive no formato não nativo, para medir a sensibilidade. A comparação principal entre versões deve respeitar também os respectivos formatos nativos. Não confundir métricas lexicais com correção factual. Não usar perguntas do conjunto de validação como benchmark independente.

In [ ]:
import sys, subprocess, os, json, csv, math, gc, random, hashlib, re, time
from pathlib import Path
from collections import Counter
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
try:
    from transformers import AutoTokenizer
except ImportError:
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', 'transformers'])
    from transformers import AutoTokenizer
from google.colab import drive
assert torch.cuda.is_available(), 'Ative GPU L4/A100 em Ambiente de execução.'
if not Path('/content/drive/MyDrive').is_dir(): drive.mount('/content/drive')
print('GPU:',torch.cuda.get_device_name(0),'| torch:',torch.__version__)

In [ ]:
ROOT=Path('/content/drive/MyDrive/Colab_LLMs/Maia_Lite_0_3_0_Chat')
BASE_CKPT=ROOT/'training_runs/maia_lite_0_3_0_a100_v3_experimental/latest.pt'
TOKENIZER_DIR=Path('/content/drive/MyDrive/Colab_LLMs/Maia_Lite_v3/tokenizer')
ADAPTERS={
    'v2.2':ROOT/'training_runs/maia_lite_0_3_0_sft_v2_2_lora_v1/adapter_best.pt',
    'v2.3':ROOT/'training_runs/maia_lite_0_3_0_sft_v2_3_lora_v1/adapter_best.pt',
}
OUTPUT=ROOT/'benchmarks/sft_v2_2_vs_v2_3'
QUICK_MODE=False # True: 1 pergunta/idioma, 2 estratégias, 2 formatos, 2 modelos = 12 gerações
MAX_NEW_TOKENS=96
SEED=20261119
TEMPERATURE=0.7; TOP_K=40; TOP_P=0.9; REPETITION_PENALTY=1.18
SAVE_EVERY_RESPONSE=True
for path in [BASE_CKPT,TOKENIZER_DIR,*ADAPTERS.values()]:
    assert path.exists(),f'Arquivo ausente: {path}'
OUTPUT.mkdir(parents=True,exist_ok=True)
print('Saída:',OUTPUT)
for name,path in ADAPTERS.items():print(name,path)

## Arquitetura MaiaGPT v3.8
As classes abaixo são reproduzidas do notebook SFT v2.3. A base é carregada **uma vez** para economizar VRAM; os adaptadores são aplicados sequencialmente e verificados por nomes, shapes e metadados.

In [ ]:
from dataclasses import dataclass
@dataclass
class Config:
    vocab_size:int=50257
    block_size:int=2048
    n_layer:int=36
    n_head:int=16
    n_embd:int=1024
    dropout:float=0.0
    bias:bool=True
cfg=Config()
class Block(nn.Module):
    def __init__(self,c):
        super().__init__()
        d=c.n_embd
        self.ln1=nn.LayerNorm(d)
        self.qkv=nn.Linear(d,3*d,bias=c.bias)
        self.proj=nn.Linear(d,d,bias=c.bias)
        self.ln2=nn.LayerNorm(d)
        self.fc=nn.Linear(d,4*d,bias=c.bias)
        self.out=nn.Linear(4*d,d,bias=c.bias)
        self.nh=c.n_head
    def forward(self,x):
        b,t,d=x.shape
        q,k,v=self.qkv(self.ln1(x)).chunk(3,dim=-1)
        q=q.view(b,t,self.nh,d//self.nh).transpose(1,2)
        k=k.view(b,t,self.nh,d//self.nh).transpose(1,2)
        v=v.view(b,t,self.nh,d//self.nh).transpose(1,2)
        a=F.scaled_dot_product_attention(q,k,v,is_causal=True,dropout_p=0.0)
        x=x+self.proj(a.transpose(1,2).contiguous().view(b,t,d))
        x=x+self.out(F.gelu(self.fc(self.ln2(x)),approximate='tanh'))
        return x
class MaiaGPT(nn.Module):
    def __init__(self,c):
        super().__init__();self.cfg=c
        self.tok=nn.Embedding(c.vocab_size,c.n_embd)
        self.pos=nn.Embedding(c.block_size,c.n_embd)
        self.blocks=nn.ModuleList([Block(c) for _ in range(c.n_layer)])
        self.norm=nn.LayerNorm(c.n_embd)
        self.head=nn.Linear(c.n_embd,c.vocab_size,bias=False)
        self.head.weight=self.tok.weight
        self.apply(self.init_weights)
    @staticmethod
    def init_weights(m):
        if isinstance(m,(nn.Linear,nn.Embedding)):
            nn.init.normal_(m.weight,mean=0,std=0.02)
            if isinstance(m,nn.Linear) and m.bias is not None:nn.init.zeros_(m.bias)
    def forward(self,idx,targets=None):
        b,t=idx.shape
        assert t<=self.cfg.block_size
        x=self.tok(idx)+self.pos(torch.arange(t,device=idx.device))
        for block in self.blocks:x=block(x)
        x=self.norm(x)
        logits=self.head(x)
        loss=None if targets is None else F.cross_entropy(logits.reshape(-1,logits.size(-1)).float(),targets.reshape(-1),ignore_index=-100)
        return logits,loss

In [ ]:
tokenizer=AutoTokenizer.from_pretrained(str(TOKENIZER_DIR),local_files_only=True,use_fast=True)
assert len(tokenizer)==cfg.vocab_size and tokenizer.eos_token_id is not None
checkpoint=torch.load(BASE_CKPT,map_location='cpu',weights_only=False) # SOMENTE checkpoint próprio confiável
assert int(checkpoint['step'])==300000
model=MaiaGPT(cfg)
model.load_state_dict(checkpoint['model'],strict=True)
del checkpoint;gc.collect()
model=model.to('cuda',dtype=torch.bfloat16)
for p in model.parameters():p.requires_grad_(False)
model.eval()
print('Base 300000 carregada; tokenizer EOS:',tokenizer.eos_token_id)

In [ ]:
class LoRALinear(nn.Module):
    def __init__(self, base, rank, alpha, dropout):
        super().__init__()
        assert isinstance(base,nn.Linear)
        self.base=base
        self.scale=alpha/rank
        self.dropout=nn.Dropout(dropout)
        self.A=nn.Parameter(torch.empty(rank,base.in_features,device=base.weight.device,dtype=torch.float32))
        self.B=nn.Parameter(torch.zeros(base.out_features,rank,device=base.weight.device,dtype=torch.float32))
        nn.init.kaiming_uniform_(self.A,a=math.sqrt(5))
    def forward(self,x):
        # Base remains BF16; LoRA computation in BF16 autocast, parameters FP32.
        update=F.linear(F.linear(self.dropout(x).float(),self.A),self.B)*self.scale
        return self.base(x)+update.to(x.dtype)


# Adaptadores devem compartilhar arquitetura e módulos alvo.
metadata={}
for version,path in ADAPTERS.items():
    payload=torch.load(path,map_location='cpu',weights_only=False) # arquivos próprios confiáveis
    assert all(k in payload for k in ('adapter','config','update'))
    config=payload['config']
    assert config['architecture']=='MaiaGPT v3.8'
    assert list(config['target_modules'])==['qkv','proj']
    assert int(config['base_step'])==300000
    metadata[version]={'step':int(payload['update']),'rank':int(config['rank']),
                       'alpha':int(config['alpha']),'dropout':float(config['dropout']),
                       'val_loss':payload.get('val_loss'),'path':str(path)}
    del payload
assert len({(x['rank'],x['alpha'],x['dropout']) for x in metadata.values()})==1, 'Configurações LoRA distintas; adaptar o loader.'
common=metadata['v2.2']
for block in model.blocks:
    block.qkv=LoRALinear(block.qkv,common['rank'],common['alpha'],common['dropout'])
    block.proj=LoRALinear(block.proj,common['rank'],common['alpha'],common['dropout'])
expected={name for name,_ in model.named_parameters() if name.endswith('.A') or name.endswith('.B')}
assert len(expected)==cfg.n_layer*4
print('Metadados:',json.dumps(metadata,indent=2,ensure_ascii=False))

def load_adapter(version):
    payload=torch.load(ADAPTERS[version],map_location='cpu',weights_only=False)
    state=payload['adapter']
    assert set(state)==expected, f'Chaves incompatíveis para {version}'
    named=dict(model.named_parameters())
    with torch.no_grad():
        for name in expected:
            assert tuple(state[name].shape)==tuple(named[name].shape),(name,state[name].shape,named[name].shape)
            named[name].copy_(state[name].to(device=named[name].device,dtype=named[name].dtype))
    model.eval()
    del payload,state;gc.collect()
    print('Adaptador ativo:',version,'step:',metadata[version]['step'])

## Perguntas externas fixas
Nove perguntas (três por idioma), sem dependência do arquivo de validação. Uma questão por idioma testa o reconhecimento de informação ausente. **Essas perguntas não devem ser usadas para treinar a versão seguinte.**

In [ ]:
QUESTIONS=[
 {'id':'pt_arvores','lang':'pt','prompt':'Responda de forma clara, objetiva e cientificamente correta. Quais são os benefícios do plantio de árvores nas cidades?','type':'factual'},
 {'id':'pt_fotossintese','lang':'pt','prompt':'Responda de forma clara, objetiva e cientificamente correta. O que é fotossíntese e por que ela é importante?','type':'factual'},
 {'id':'pt_sem_artigo','lang':'pt','prompt':'Se não souber, indique a incerteza. Quais são as duas técnicas exatas de decomposição celular mencionadas em um artigo que não foi fornecido?','type':'uncertainty'},
 {'id':'en_kanji','lang':'en','prompt':'Answer clearly, concisely, and accurately. How does the use of Chinese characters in Japanese differ from their use in Chinese languages?','type':'factual'},
 {'id':'en_gravity','lang':'en','prompt':'Answer clearly and accurately. Why do objects fall toward Earth?','type':'factual'},
 {'id':'en_missing','lang':'en','prompt':'If information is missing, acknowledge uncertainty. What exact numerical accuracy did a research paper achieve if the paper has not been provided?','type':'uncertainty'},
 {'id':'es_arboles','lang':'es','prompt':'Responde de forma clara, concisa y precisa. ¿Cuáles son las ventajas de plantar árboles para la población?','type':'factual'},
 {'id':'es_agua','lang':'es','prompt':'Responde de forma clara y precisa. ¿Por qué es importante el agua para los seres vivos?','type':'factual'},
 {'id':'es_sin_articulo','lang':'es','prompt':'Si falta información, reconoce la incertidumbre. ¿Cuáles fueron los dos resultados exactos de un artículo científico que no se ha proporcionado?','type':'uncertainty'},
]
if QUICK_MODE:QUESTIONS=[next(q for q in QUESTIONS if q['lang']==lang) for lang in ('pt','en','es')]
STRATEGIES=['greedy','topk_topp','topk_topp_rep'] if not QUICK_MODE else ['greedy','topk_topp_rep']
FORMATS=['original','newline']
def format_prompt(prompt,fmt):
    assert fmt in FORMATS
    return prompt.rstrip() + ('\n' if fmt=='newline' else '')
print('Total previsto:',len(QUESTIONS)*len(STRATEGIES)*len(FORMATS)*len(ADAPTERS))

In [ ]:
def set_seed(seed):
    random.seed(seed);np.random.seed(seed % (2**32-1));torch.manual_seed(seed)
    if torch.cuda.is_available():torch.cuda.manual_seed_all(seed)

def sample_token(logits, strategy, prior_ids):
    scores=logits.float().clone()
    if strategy=='greedy':return int(torch.argmax(scores).item())
    if strategy=='topk_topp_rep':
        # Penalidade aplicada aos tokens já gerados (não ao prompt).
        for tid in set(prior_ids):
            scores[tid]=scores[tid]/REPETITION_PENALTY if scores[tid]>0 else scores[tid]*REPETITION_PENALTY
    scores=scores/TEMPERATURE
    if TOP_K>0:
        kth=torch.topk(scores,min(TOP_K,scores.numel())).values[-1]
        scores[scores<kth]=-float('inf')
    if TOP_P<1.0:
        sorted_scores,idx=torch.sort(scores,descending=True)
        probs=torch.softmax(sorted_scores,dim=-1)
        remove=(torch.cumsum(probs,dim=-1)-probs)>TOP_P
        sorted_scores[remove]=-float('inf')
        scores=torch.full_like(scores,-float('inf')).scatter_(0,idx,sorted_scores)
    return int(torch.multinomial(torch.softmax(scores,dim=-1),1).item())

@torch.inference_mode()
def generate_response(prompt,fmt,strategy,seed,max_new_tokens=MAX_NEW_TOKENS):
    set_seed(seed)
    model.eval()
    formatted=format_prompt(prompt,fmt)
    input_ids=tokenizer(formatted,add_special_tokens=False,return_tensors='pt')['input_ids'].to('cuda')
    assert input_ids.numel()>0
    ids=input_ids
    new_ids=[];ended_eos=False
    for _ in range(max_new_tokens):
        if ids.shape[1]>=cfg.block_size:break
        with torch.autocast('cuda',dtype=torch.bfloat16):logits,_=model(ids)
        tid=sample_token(logits[0,-1,:],strategy,new_ids)
        if tid==tokenizer.eos_token_id:
            ended_eos=True;break
        new_ids.append(tid)
        ids=torch.cat((ids,torch.tensor([[tid]],device='cuda')),dim=1)
    response=tokenizer.decode(new_ids,skip_special_tokens=True)
    return {'response':response,'new_tokens':len(new_ids),'ended_eos':ended_eos,
            'first_token_id':new_ids[0] if new_ids else None,
            'prompt_tokens':int(input_ids.numel())}

## Execução e retomada
Salva cada resposta em `results.jsonl` com escrita incremental. A chave inclui modelo, pergunta, formato e estratégia; uma execução interrompida continua de onde parou. O hash da configuração impede misturar resultados de experimentos distintos. `greedy` é determinístico; para amostragem, a seed é idêntica nos pares, mas a distribuição de cada modelo pode produzir tokens diferentes.

In [ ]:
RUN_CONFIG={'questions':QUESTIONS,'strategies':STRATEGIES,'formats':FORMATS,'max_new_tokens':MAX_NEW_TOKENS,
            'temperature':TEMPERATURE,'top_k':TOP_K,'top_p':TOP_P,'repetition_penalty':REPETITION_PENALTY,
            'seed':SEED,'adapters':metadata,'base_checkpoint':str(BASE_CKPT)}
RUN_ID=hashlib.sha256(json.dumps(RUN_CONFIG,sort_keys=True,ensure_ascii=False).encode()).hexdigest()[:16]
CONFIG_FILE=OUTPUT/'run_config.json';RESULTS_FILE=OUTPUT/'results.jsonl'
if CONFIG_FILE.exists():
    old=json.loads(CONFIG_FILE.read_text(encoding='utf8'))
    assert old['run_id']==RUN_ID, 'Configuração mudou. Use outro OUTPUT ou arquive resultados anteriores.'
else:
    CONFIG_FILE.write_text(json.dumps({'run_id':RUN_ID,**RUN_CONFIG},indent=2,ensure_ascii=False),encoding='utf8')
rows=[]
if RESULTS_FILE.exists():
    with RESULTS_FILE.open(encoding='utf8') as f:
        rows=[json.loads(line) for line in f if line.strip()]
assert all(r['run_id']==RUN_ID for r in rows)
def row_key(r):return (r['model'],r['question_id'],r['format'],r['strategy'])
done={row_key(r) for r in rows}
print('Run:',RUN_ID,'| respostas existentes:',len(done))
for version in ADAPTERS:
    todo=[(q,fmt,strategy) for q in QUESTIONS for fmt in FORMATS for strategy in STRATEGIES
          if (version,q['id'],fmt,strategy) not in done]
    if not todo:continue
    load_adapter(version)
    for j,(q,fmt,strategy) in enumerate(todo,1):
        # Seed pareada por questão/formato/estratégia, independente do modelo
        seed=int(hashlib.sha256(f"{SEED}|{q['id']}|{fmt}|{strategy}".encode()).hexdigest()[:8],16)
        generated=generate_response(q['prompt'],fmt,strategy,seed)
        record={'run_id':RUN_ID,'model':version,'question_id':q['id'],'language':q['lang'],
                'type':q['type'],'prompt':q['prompt'],'format':fmt,'strategy':strategy,
                'seed':seed,**generated}
        with RESULTS_FILE.open('a',encoding='utf8') as f:
            f.write(json.dumps(record,ensure_ascii=False)+'\n');f.flush()
        rows.append(record);done.add(row_key(record))
        print(f"[{version} {j}/{len(todo)}] {q['id']} {fmt} {strategy}: {generated['new_tokens']} tokens, EOS={generated['ended_eos']}")
print('Concluído:',len(rows),'respostas')

## Métricas automáticas e avaliação humana cega
`repeat_trigram` usa tokens lexicais Unicode e mede redundância de trigramas, não correção factual. `EOS_rate` mede encerramento, não qualidade. A avaliação cega é necessária para relevância, coerência, precisão e reconhecimento de incerteza.

In [ ]:
import pandas as pd

def metrics(text):
    words=re.findall(r'[^\W_]+',text.lower(),flags=re.UNICODE)
    grams=list(zip(words,words[1:],words[2:]))
    repeat=(len(grams)-len(set(grams)))/len(grams) if grams else 0.0
    return {'words':len(words),'repeat_trigram':repeat,'unique_word_ratio':len(set(words))/len(words) if words else 0.0}
full=pd.DataFrame([{**r,**metrics(r['response'])} for r in rows])
full.to_csv(OUTPUT/'generations.csv',index=False,encoding='utf-8-sig')
summary=(full.groupby(['model','format','strategy','language'],dropna=False)
    .agg(n=('response','size'),mean_tokens=('new_tokens','mean'),eos_rate=('ended_eos','mean'),
         mean_repeat_trigram=('repeat_trigram','mean'),mean_unique_word_ratio=('unique_word_ratio','mean'),
         empty_rate=('new_tokens',lambda x:float((x==0).mean())))
    .reset_index())
summary.to_csv(OUTPUT/'automatic_summary.csv',index=False,encoding='utf-8-sig')
paired=full.pivot(index=['question_id','language','format','strategy','prompt'],columns='model',values=['response','new_tokens','ended_eos','repeat_trigram'])
paired.columns=[f'{metric}_{version}' for metric,version in paired.columns]
paired=paired.reset_index()
paired.to_csv(OUTPUT/'paired_responses.csv',index=False,encoding='utf-8-sig')
print('Resumo automático:')
display(summary)
print('Arquivos:',OUTPUT/'automatic_summary.csv',OUTPUT/'paired_responses.csv')

In [ ]:
# Ficha cega: identificadores A/B embaralhados por comparação pareada
rng=random.Random(SEED)
blind=[];key=[]
for _,r in paired.iterrows():
    versions=['v2.2','v2.3'];rng.shuffle(versions)
    bid=hashlib.sha256(f"{RUN_ID}|{r['question_id']}|{r['format']}|{r['strategy']}".encode()).hexdigest()[:12]
    for letter,version in zip(('A','B'),versions):
        blind.append({'pair_id':bid,'option':letter,'question_id':r['question_id'],'language':r['language'],
                      'format':r['format'],'strategy':r['strategy'],'prompt':r['prompt'],
                      'response':r[f'response_{version}'],
                      'relevance_1_5':'','coherence_1_5':'','factuality_1_5':'',
                      'uncertainty_1_5_or_na':'','repetition_1_5':'','notes':''})
        key.append({'pair_id':bid,'option':letter,'model':version})
pd.DataFrame(blind).to_csv(OUTPUT/'human_review_blind.csv',index=False,encoding='utf-8-sig')
pd.DataFrame(key).to_csv(OUTPUT/'blind_key.csv',index=False,encoding='utf-8-sig')
print('Avaliação cega:',OUTPUT/'human_review_blind.csv')
print('Chave confidencial:',OUTPUT/'blind_key.csv')
print('ATENÇÃO: não consulte blind_key.csv durante a avaliação humana.')

## Como interpretar

1. Compare **v2.2/original** com **v2.3/newline** para estimar o desempenho dos sistemas nos formatos nativos.
2. Compare **v2.2/original** com **v2.2/newline** e **v2.3/original** com **v2.3/newline** para medir sensibilidade ao formato.
3. Compare **v2.2/newline** com **v2.3/newline** para isolar parcialmente o efeito do adaptador sob entrada idêntica (a diferença de dados e comprimento de contexto permanece).
4. Analise `greedy` separadamente das estratégias amostradas; penalização de repetição pode diminuir redundância sem melhorar a verdade factual.
5. Avalie cegamente respostas completas. **Não use `blind_key.csv` até finalizar as notas.**

**Saídas:** `run_config.json`, `results.jsonl`, `generations.csv`, `automatic_summary.csv`, `paired_responses.csv`, `human_review_blind.csv`, `blind_key.csv`. Nenhum checkpoint é modificado.